<a href="https://colab.research.google.com/github/kimmy111-zhu/human-validation/blob/main/notebooks/ifeval_esl_matched_stratified_sampling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
import random
import os
import pandas as pd
from google.colab import files


# =========================================================
# 1. Settings
# =========================================================

RANDOM_SEED = 42
SAMPLE_SIZE = 5

BENCHMARK_NAME = "IFEval"


# =========================================================
# 2. Language files
# =========================================================

FILE_PATHS = {
    "Arabic": "/content/arabic.jsonl",
    "French": "/content/french.jsonl",
    "German": "/content/german.jsonl",
    "Japanese": "/content/japanese.jsonl",
    "Mandarin": "/content/mandarin.jsonl",
    "Portuguese": "/content/portuguese.jsonl",
    "Russian": "/content/russian.jsonl",
    "Spanish": "/content/spanish.jsonl"
}

LANGUAGES = list(FILE_PATHS.keys())


# =========================================================
# 3. Check input files
# =========================================================

print("Checking input files:\n")

for language, file_path in FILE_PATHS.items():

    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"File not found: {file_path}\n"
            f"Please upload the {language} file to Colab."
        )

    print(f"{language}: {file_path}")


# =========================================================
# 4. Helper functions
# =========================================================

def read_jsonl(file_path):
    """
    Read one JSONL file and preserve the original row number.
    """

    records = []

    with open(file_path, "r", encoding="utf-8-sig") as file:

        for line_number, line in enumerate(file, start=1):

            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)

            except json.JSONDecodeError as error:
                raise ValueError(
                    f"Invalid JSON on line {line_number}\n"
                    f"File: {file_path}\n"
                    f"Error: {error}"
                )

            record["_source_row"] = line_number
            records.append(record)

    return records


def normalize_text(value):
    """
    Normalize spaces and line breaks only for comparison.

    The original text written to the CSV is not changed.
    """

    if value is None:
        return ""

    return " ".join(str(value).split())


def convert_to_cell(value):
    """
    Convert lists and dictionaries into readable CSV text.
    """

    if value is None:
        return ""

    if isinstance(value, (list, dict)):
        return json.dumps(
            value,
            ensure_ascii=False
        )

    return str(value)


# =========================================================
# 5. Read all 8 language files
# =========================================================

datasets = {}

print("\nLoading files:\n")

for language, file_path in FILE_PATHS.items():

    datasets[language] = read_jsonl(file_path)

    print(
        f"Loaded {len(datasets[language])} records "
        f"for {language}"
    )


# Show field names for checking
print("\nFields found in the first record of each file:")

for language in LANGUAGES:

    if datasets[language]:

        print(
            f"\n{language}: "
            f"{list(datasets[language][0].keys())}"
        )


# =========================================================
# 6. Build matching maps using IFEval 'key'
# =========================================================

record_maps = {}

print("\nBuilding matching maps:")

for language, records in datasets.items():

    current_map = {}

    missing_key_count = 0
    missing_prompt_count = 0
    duplicate_key_count = 0

    for record in records:

        item_key = record.get("key", None)
        original_prompt = record.get("prompt", "")

        if item_key is None or str(item_key).strip() == "":
            missing_key_count += 1
            continue

        if not normalize_text(original_prompt):
            missing_prompt_count += 1
            continue

        # Convert key to string for consistent matching
        key_string = str(item_key)

        if key_string in current_map:
            duplicate_key_count += 1
            continue

        current_map[key_string] = record

    record_maps[language] = current_map

    print(
        f"\n{language}: "
        f"{len(current_map)} usable records"
    )

    if missing_key_count > 0:
        print(
            f"Warning: {missing_key_count} records "
            f"had no usable key."
        )

    if missing_prompt_count > 0:
        print(
            f"Warning: {missing_prompt_count} records "
            f"had no usable prompt."
        )

    if duplicate_key_count > 0:
        print(
            f"Warning: {duplicate_key_count} duplicate keys "
            f"were found. The first record was kept."
        )


# =========================================================
# 7. Find keys shared by all 8 languages
# =========================================================

first_language = LANGUAGES[0]

common_keys = set(
    record_maps[first_language].keys()
)

for language in LANGUAGES[1:]:

    common_keys &= set(
        record_maps[language].keys()
    )


print(
    f"\nCommon keys across all "
    f"{len(LANGUAGES)} languages: "
    f"{len(common_keys)}"
)


# =========================================================
# 8. Verify that the original prompt is the same
#    across all languages for each shared key
# =========================================================

valid_matched_keys = []
prompt_mismatch_keys = []

for item_key in common_keys:

    prompts_for_key = []

    for language in LANGUAGES:

        prompt = record_maps[language][item_key].get(
            "prompt",
            ""
        )

        prompts_for_key.append(
            normalize_text(prompt)
        )

    # All 8 original prompts must be identical
    if len(set(prompts_for_key)) == 1:
        valid_matched_keys.append(item_key)

    else:
        prompt_mismatch_keys.append(item_key)


# Sort before sampling for reproducibility
valid_matched_keys = sorted(
    valid_matched_keys,
    key=lambda value: str(value)
)


print(
    f"Keys with identical original prompts "
    f"across all languages: "
    f"{len(valid_matched_keys)}"
)

print(
    f"Keys excluded because prompts differed: "
    f"{len(prompt_mismatch_keys)}"
)


if len(valid_matched_keys) < SAMPLE_SIZE:

    raise ValueError(
        f"Only {len(valid_matched_keys)} fully matched keys "
        f"were found across all language files.\n"
        f"Cannot sample {SAMPLE_SIZE} prompts."
    )


# =========================================================
# 9. Randomly sample 5 fully matched keys
# =========================================================

random.seed(RANDOM_SEED)

selected_keys = random.sample(
    valid_matched_keys,
    SAMPLE_SIZE
)


print(f"\nRandom seed: {RANDOM_SEED}")

print(
    f"Selected matched keys: "
    f"{len(selected_keys)}"
)

print(f"Selected keys: {selected_keys}")


# =========================================================
# 10. Create matched stratified output
# =========================================================

output_rows = []

for prompt_number, item_key in enumerate(
    selected_keys,
    start=1
):

    base_id = (
        f"{BENCHMARK_NAME}_ESL_{prompt_number:03d}"
    )

    # Original prompt should be identical across all languages
    reference_record = record_maps[first_language][
        item_key
    ]

    original_prompt = reference_record.get(
        "prompt",
        ""
    )

    for language in LANGUAGES:

        record = record_maps[language][item_key]

        modified_prompt = record.get(
            "text_transformed",
            ""
        )

        instruction_ids = record.get(
            "instruction_id_list",
            []
        )

        instruction_kwargs = record.get(
            "kwargs",
            []
        )

        applied_rules = record.get(
            "applied_rules",
            []
        )

        output_rows.append({

            "Base_ID": base_id,

            "Sample_ID": (
                f"{base_id}_{language}"
            ),

            "Benchmark": BENCHMARK_NAME,

            "Language": language,

            "Key": item_key,

            "Source_File": os.path.basename(
                FILE_PATHS[language]
            ),

            "Source_Row": record.get(
                "_source_row",
                ""
            ),

            "Original_Prompt": convert_to_cell(
                original_prompt
            ),

            "Modified_Prompt": convert_to_cell(
                modified_prompt
            ),

            "Instruction_IDs": convert_to_cell(
                instruction_ids
            ),

            "Instruction_Kwargs": convert_to_cell(
                instruction_kwargs
            ),

            "Applied_Rules": convert_to_cell(
                applied_rules
            ),

            # Meaning preservation
            "R1_Meaning": "",
            "R2_Meaning": "",
            "Final_Meaning": "",

            # Key information / instruction constraint preservation
            "R1_Key_Info": "",
            "R2_Key_Info": "",
            "Final_Key_Info": "",

            # ESL realism
            "R1_Realism": "",
            "R2_Realism": "",
            "Final_Realism": "",

            # Readability
            "R1_Readability": "",
            "R2_Readability": "",
            "Final_Readability": "",

            # General comments
            "Comments": ""
        })


# =========================================================
# 11. Convert to DataFrame
# =========================================================

sample_df = pd.DataFrame(
    output_rows
)


print("\nSampling completed.")

print(
    f"Selected original prompts: "
    f"{SAMPLE_SIZE}"
)

print(
    f"Number of language strata: "
    f"{len(LANGUAGES)}"
)

print(
    f"Total output rows: "
    f"{len(sample_df)}"
)


display(sample_df)


# =========================================================
# 12. Check important columns
# =========================================================

empty_original = (
    sample_df["Original_Prompt"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

empty_modified = (
    sample_df["Modified_Prompt"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

empty_instruction_ids = (
    sample_df["Instruction_IDs"]
    .astype(str)
    .str.strip()
    .isin(["", "[]"])
    .sum()
)

empty_instruction_kwargs = (
    sample_df["Instruction_Kwargs"]
    .astype(str)
    .str.strip()
    .isin(["", "[]"])
    .sum()
)


print("\nColumn check:")

print(
    f"Empty Original_Prompt rows: "
    f"{empty_original}"
)

print(
    f"Empty Modified_Prompt rows: "
    f"{empty_modified}"
)

print(
    f"Empty Instruction_IDs rows: "
    f"{empty_instruction_ids}"
)

print(
    f"Empty Instruction_Kwargs rows: "
    f"{empty_instruction_kwargs}"
)


# =========================================================
# 13. Check language stratification
# =========================================================

print("\nRows per language stratum:")

language_counts = (
    sample_df["Language"]
    .value_counts()
    .reindex(LANGUAGES)
)

print(language_counts)


# =========================================================
# 14. Check matched sampling
# =========================================================

# Each Base_ID should appear exactly 8 times
base_id_counts = (
    sample_df
    .groupby("Base_ID")
    .size()
)

incorrect_base_ids = base_id_counts[
    base_id_counts != len(LANGUAGES)
]


if len(incorrect_base_ids) == 0:

    print(
        "\nMatched sampling check passed: "
        "every Base_ID has all 8 language versions."
    )

else:

    print(
        "\nWarning: Some Base_ID values do not have "
        "exactly 8 language versions:"
    )

    print(incorrect_base_ids)


# Check that every Base_ID contains all expected languages
expected_languages = set(LANGUAGES)

language_set_check = (
    sample_df
    .groupby("Base_ID")["Language"]
    .apply(lambda values: set(values))
)

incorrect_language_sets = language_set_check[
    language_set_check.apply(
        lambda values: values != expected_languages
    )
]


if len(incorrect_language_sets) == 0:

    print(
        "Language check passed: every Base_ID contains "
        "Arabic, French, German, Japanese, Mandarin, "
        "Portuguese, Russian, and Spanish."
    )

else:

    print(
        "\nWarning: Some Base_ID values are missing "
        "one or more language versions:"
    )

    print(incorrect_language_sets)


# =========================================================
# 15. Check that every language has exactly 5 rows
# =========================================================

incorrect_language_counts = language_counts[
    language_counts != SAMPLE_SIZE
]


if len(incorrect_language_counts) == 0:

    print(
        f"Stratification check passed: every language "
        f"has exactly {SAMPLE_SIZE} rows."
    )

else:

    print(
        "\nWarning: Some language strata do not have "
        f"exactly {SAMPLE_SIZE} rows:"
    )

    print(incorrect_language_counts)


# =========================================================
# 16. Check that the original prompt is identical
#     within each Base_ID
# =========================================================

prompt_count_per_base = (
    sample_df
    .groupby("Base_ID")["Original_Prompt"]
    .nunique()
)

prompt_mismatch_in_output = prompt_count_per_base[
    prompt_count_per_base != 1
]


if len(prompt_mismatch_in_output) == 0:

    print(
        "Original-prompt check passed: "
        "each Base_ID has one identical original prompt."
    )

else:

    print(
        "\nWarning: Some Base_ID values contain "
        "different original prompts:"
    )

    print(prompt_mismatch_in_output)


# =========================================================
# 17. Save and download CSV
# =========================================================

output_file = (
    "/content/"
    "IFEval_ESL_matched_stratified_"
    "sample_n5_per_language_seed42.csv"
)


sample_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)


print(
    f"\nCSV saved successfully: "
    f"{output_file}"
)


files.download(output_file)

Checking input files:

Arabic: /content/arabic.jsonl
French: /content/french.jsonl
German: /content/german.jsonl
Japanese: /content/japanese.jsonl
Mandarin: /content/mandarin.jsonl
Portuguese: /content/portuguese.jsonl
Russian: /content/russian.jsonl
Spanish: /content/spanish.jsonl

Loading files:

Loaded 541 records for Arabic
Loaded 541 records for French
Loaded 541 records for German
Loaded 541 records for Japanese
Loaded 541 records for Mandarin
Loaded 541 records for Portuguese
Loaded 541 records for Russian
Loaded 541 records for Spanish

Fields found in the first record of each file:

Arabic: ['key', 'prompt', 'instruction_id_list', 'kwargs', 'text_transformed', 'applied_rules', '_source_row']

French: ['key', 'prompt', 'instruction_id_list', 'kwargs', 'text_transformed', 'applied_rules', '_source_row']

German: ['key', 'prompt', 'instruction_id_list', 'kwargs', 'text_transformed', 'applied_rules', '_source_row']

Japanese: ['key', 'prompt', 'instruction_id_list', 'kwargs', 'tex

,Base_ID,Sample_ID,Benchmark,Language,Key,Source_File,Source_Row,Original_Prompt,Modified_Prompt,Instruction_IDs,...,R1_Key_Info,R2_Key_Info,Final_Key_Info,R1_Realism,R2_Realism,Final_Realism,R1_Readability,R2_Readability,Final_Readability,Comments
0,IFEval_ESL_001,IFEval_ESL_001_Arabic,IFEval,Arabic,1620,arabic.jsonl,115,I really love the album called Lilith. I want ...,I really love the album called Lilith. I want ...,"[""detectable_content:postscript""]",...,,,,,,,,,,
1,IFEval_ESL_001,IFEval_ESL_001_French,IFEval,French,1620,french.jsonl,115,I really love the album called Lilith. I want ...,You want to introduce Lilith to Luheng? Draft ...,"[""detectable_content:postscript""]",...,,,,,,,,,,
2,IFEval_ESL_001,IFEval_ESL_001_German,IFEval,German,1620,german.jsonl,115,I really love the album called Lilith. I want ...,Introduce the album Lilith to my friend Luheng...,"[""detectable_content:postscript""]",...,,,,,,,,,,
3,IFEval_ESL_001,IFEval_ESL_001_Japanese,IFEval,Japanese,1620,japanese.jsonl,115,I really love the album called Lilith. I want ...,I really love the album called Lilith. I will ...,"[""detectable_content:postscript""]",...,,,,,,,,,,
4,IFEval_ESL_001,IFEval_ESL_001_Mandarin,IFEval,Mandarin,1620,mandarin.jsonl,115,I really love the album called Lilith. I want ...,"Lilith, the album I love, I want to introduce ...","[""detectable_content:postscript""]",...,,,,,,,,,,
5,IFEval_ESL_001,IFEval_ESL_001_Portuguese,IFEval,Portuguese,1620,portuguese.jsonl,115,I really love the album called Lilith. I want ...,I really loved the album called Lilith because...,"[""detectable_content:postscript""]",...,,,,,,,,,,
6,IFEval_ESL_001,IFEval_ESL_001_Russian,IFEval,Russian,1620,russian.jsonl,115,I really love the album called Lilith. I want ...,"I want to introduce the best album, Lilith, to...","[""detectable_content:postscript""]",...,,,,,,,,,,
7,IFEval_ESL_001,IFEval_ESL_001_Spanish,IFEval,Spanish,1620,spanish.jsonl,115,I really love the album called Lilith. I want ...,Could you draft an email introducing the album...,"[""detectable_content:postscript""]",...,,,,,,,,,,
8,IFEval_ESL_002,IFEval_ESL_002_Arabic,IFEval,Arabic,1132,arabic.jsonl,26,Write the lyrics to a hit song by the rock ban...,Write the lyrics to a hit song by the rock ban...,"[""change_case:english_capital"", ""keywords:forb...",...,,,,,,,,,,
9,IFEval_ESL_002,IFEval_ESL_002_French,IFEval,French,1132,french.jsonl,26,Write the lyrics to a hit song by the rock ban...,Write the lyrics to a popular song by the rock...,"[""change_case:english_capital"", ""keywords:forb...",...,,,,,,,,,,



Column check:
Empty Original_Prompt rows: 0
Empty Modified_Prompt rows: 0
Empty Instruction_IDs rows: 0
Empty Instruction_Kwargs rows: 0

Rows per language stratum:
Language
Arabic        5
French        5
German        5
Japanese      5
Mandarin      5
Portuguese    5
Russian       5
Spanish       5
Name: count, dtype: int64

Matched sampling check passed: every Base_ID has all 8 language versions.
Language check passed: every Base_ID contains Arabic, French, German, Japanese, Mandarin, Portuguese, Russian, and Spanish.
Stratification check passed: every language has exactly 5 rows.
Original-prompt check passed: each Base_ID has one identical original prompt.

CSV saved successfully: /content/IFEval_ESL_matched_stratified_sample_n5_per_language_seed42.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>